In [78]:
import sys
import glob
import os
from os.path import join
sys.path.append('../classifier')
import pandas as pd
import numpy as np
from torchvision import transforms
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from chexpert_binary_classifier import CheXpertClassifier
from tqdm import tqdm

STYLE = "pleural_effusion"
OTHER_STYLE = "support_devices"
IMAGE_FOLDER = f'/usr/local/data/zahrat/workshop/dent_output/interpolations/{STYLE}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the model

if STYLE == 'support_devices':
    checkpoint_1 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_support-devices.pt/best_model_512_2025-02-21_18-06-55.pth')
    checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_pleural-effusion.pt/best_model_512_2025-02-21_18-05-59.pth')
elif STYLE == 'pleural_effusion':
    checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_support-devices.pt/best_model_512_2025-02-21_18-06-55.pth')
    checkpoint_1 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_pleural-effusion.pt/best_model_512_2025-02-21_18-05-59.pth')
else:
    raise ValueError(f"Invalid style: {STYLE}")

model_1 = CheXpertClassifier()
model_1.load_state_dict(checkpoint_1['model_state_dict'])
model_1.eval()
model_1.to(DEVICE)

model_2 = CheXpertClassifier()
model_2.load_state_dict(checkpoint_2['model_state_dict'])
model_2.eval()
model_2.to(DEVICE)


/tmp/ipykernel_1969493/6448423.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_suppor

CheXpertClassifier(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (sc

In [79]:
class CustomDataset(Dataset):
    def __init__(self, img_paths, transform=None):
        self.img_paths = img_paths 
        self.transform = transform
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, img_path

    def __len__(self):
        return len(self.img_paths)

transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


seeds = os.listdir(IMAGE_FOLDER)
seeds = [int(seed.split('seed')[1]) for seed in seeds]

frames = range(0, 50)

style_image_paths = [join(IMAGE_FOLDER, f"seed{seed}", f'bezier_{seed}_{frame}.png') for seed in seeds for frame in frames]
style_image_paths = [path for path in style_image_paths if os.path.exists(path)]

# Load the dataset
dataset = CustomDataset(style_image_paths, transform=transform)
test_loader = DataLoader(dataset, batch_size=64, num_workers=8, 
                           pin_memory=True)

In [80]:
patients_info = {}

for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
    img = img.to(DEVICE)
    with torch.no_grad():
        output = model_1(img)
    output = torch.sigmoid(output)
    output = output.cpu().numpy()
    for j, path in enumerate(img_path):
        seed = path.split('/')[-1].split('_')[1]
        frame = path.split('/')[-1].split('_')[2].split('.')[0]
        
        if seed not in patients_info:
            patients_info[seed] = {'seed': seed}
            
        patients_info[seed][f'{frame}_{STYLE}'] = output[j][0]

100%|██████████| 333/333 [00:58<00:00,  5.68it/s]


In [81]:
for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
    img = img.to(DEVICE)
    with torch.no_grad():
        output = model_2(img)
    output = torch.sigmoid(output)
    output = output.cpu().numpy()
    for j, path in enumerate(img_path):
        seed = path.split('/')[-1].split('_')[1]
        frame = path.split('/')[-1].split('_')[2].split('.')[0]
        
        if seed not in patients_info:
            patients_info[seed] = {'seed': seed}
            
        patients_info[seed][f'{frame}_{OTHER_STYLE}'] = output[j][0]

100%|██████████| 333/333 [00:56<00:00,  5.86it/s]


In [82]:
df = pd.DataFrame.from_records(list(patients_info.values()), index='seed')
df.to_csv(f'../metrics/{STYLE}_interpolation_predictions.csv', index=True)

In [83]:
df.head()

,0_pleural_effusion,1_pleural_effusion,2_pleural_effusion,3_pleural_effusion,4_pleural_effusion,5_pleural_effusion,6_pleural_effusion,7_pleural_effusion,8_pleural_effusion,9_pleural_effusion,...,40_support_devices,41_support_devices,42_support_devices,43_support_devices,44_support_devices,45_support_devices,46_support_devices,47_support_devices,48_support_devices,49_support_devices
seed,,,,,,,,,,,,,,,,,,,,,
1831,0.049394,0.071840,0.125670,0.223543,0.393439,0.596840,0.711738,0.752677,0.779859,0.792125,...,0.084251,0.083819,0.088759,0.083920,0.096556,0.084783,0.077216,0.084002,0.089854,0.168965
1758,0.021214,0.026867,0.034849,0.035428,0.036639,0.038622,0.038503,0.037394,0.039136,0.038946,...,0.119334,0.125313,0.128630,0.141402,0.150335,0.157887,0.189876,0.199048,0.137167,0.089110
1900,0.019852,0.026546,0.033136,0.035007,0.038723,0.040622,0.039696,0.039705,0.040399,0.042178,...,0.932281,0.929243,0.923399,0.913763,0.914757,0.903372,0.900971,0.912701,0.921988,0.937520
1796,0.030615,0.041016,0.056586,0.084410,0.099397,0.115485,0.120729,0.109976,0.104780,0.089260,...,0.145587,0.169273,0.237689,0.297036,0.412978,0.516476,0.522405,0.633280,0.686503,0.778222
1823,0.108168,0.148369,0.164628,0.177314,0.187644,0.198603,0.201430,0.217177,0.219891,0.231127,...,0.067965,0.069307,0.064213,0.066361,0.063991,0.066699,0.057090,0.057277,0.049403,0.043407


# Analysis 

### Analysing for Support Devices

In [75]:
df = pd.read_csv(f'../metrics/support_devices_interpolation_predictions.csv', index_col='seed')

In [76]:
pe_cols = [f'{i}_pleural_effusion' for i in range(30, 50)]
pe_traj = df[pe_cols].sub(df['0_pleural_effusion'], axis=0).mean(axis=1)


sd_sols = [f'{i}_support_devices' for i in range(30, 50)]
sd_traj = df[sd_sols].sub(df['0_support_devices'], axis=0).mean(axis=1)



In [77]:
(sd_traj > pe_traj).sum() / len(sd_traj)

0.8632432432432432

### Analysing for Pleural Effusion

In [84]:
df = pd.read_csv(f'../metrics/pleural_effusion_interpolation_predictions.csv', index_col='seed')

pe_cols = [f'{i}_pleural_effusion' for i in range(30, 50)]
pe_traj = df[pe_cols].sub(df['0_pleural_effusion'], axis=0).mean(axis=1)


sd_sols = [f'{i}_support_devices' for i in range(30, 50)]
sd_traj = df[sd_sols].sub(df['0_support_devices'], axis=0).mean(axis=1)



In [85]:
(pe_traj - sd_traj > 0).sum() / df.shape[0]

0.7300469483568075